In [ ]:

import pandas as pd
import numpy as np
import lightgbm as lgm
import joblib
import gc
import os
import pickle
import warnings
warnings.filterwarnings("ignore")

OTHER = r"D:\M5 Forecasting"
SUFFIX = "_evaluation"    

STORES = ['CA_1','CA_2','CA_3','CA_4','TX_1','TX_2','TX_3','WI_1','WI_2','WI_3']

D_SHIFT_LIST = [1, 7]
D_WINDOW_LIST = [7, 14, 30]
DYNAMIC_COLS = [f"rolling_mean_{s}_{w}" for s in D_SHIFT_LIST for w in D_WINDOW_LIST]

HORIZON = 28
HIST_BUFFER = 100

os.makedirs(OTHER + r"\Submission", exist_ok=True)


def rmsse_from_scale(y_true, y_pred, scale):
    if scale is None or np.isnan(scale) or scale == 0:
        return np.nan
    mse = np.mean((y_true - y_pred) ** 2)
    return np.sqrt(mse / scale)

In [ ]:

def run_recursive_rollout(model, sales, testdata, feature_cols, dyn_cols,
                           d_shift_list=D_SHIFT_LIST, d_window_list=D_WINDOW_LIST,
                           hist_buffer=HIST_BUFFER):
    last_real_day = sales["Date"].max()
    print(last_real_day)
    last_day = sales[sales["Date"] == last_real_day][["item_id"] + dyn_cols].set_index("item_id")
    hist = sales[sales["Date"] > (last_real_day - hist_buffer)][["item_id", "Date", "sales"]]
    future = testdata[["item_id", "Date"]].copy()
    future["sales"] = np.nan
    buf = pd.concat([hist, future])
    del hist, future
    valid_dates = sorted(testdata["Date"].unique())
    all_preds = []
    for day_idx, current_date in enumerate(valid_dates, start=1):
        x = testdata[testdata["Date"] == current_date].set_index("item_id", drop=False).copy()
        missing = x.index.difference(last_day.index)
        if len(missing) > 0:
            x = x.drop(index=missing)
        if len(x) == 0:
            print(f"  WARNING: day {current_date} has zero rows after filtering — skipping")
            continue
        g = buf.groupby("item_id")["sales"]
        for d_shift in d_shift_list:
            for d_window in d_window_list:
                col_name = f"rolling_mean_{d_shift}_{d_window}"
                feat = g.transform(lambda s: s.shift(d_shift).rolling(d_window).mean())
                temp = buf.assign(**{col_name: feat})
                temp = temp[temp["Date"] == current_date].set_index("item_id")[col_name]
                x[col_name] = temp.reindex(x.index).values
        missing_cols = set(feature_cols) - set(x.columns)
        if missing_cols:
            raise ValueError(f"Day {current_date}: x is missing columns {missing_cols} before predict()")
        preds = model.predict(x[feature_cols])
        preds = np.clip(preds, 0, None)
        preds = np.round(preds, 0).astype(float)
        x["sales"] = preds
        all_preds.append(x[["item_id", "store_id", "Date", "sales"]].reset_index(drop=True))
        buf.loc[buf["Date"] == current_date, "sales"] = (buf.loc[buf["Date"] == current_date, "item_id"].map(x.set_index("item_id")["sales"]).to_numpy())
    return pd.concat(all_preds, ignore_index=True)

In [ ]:
recursive_all = []

for i in STORES:
    print("RECURSIVE PREDICT:", i)

    testdata = pd.read_pickle(OTHER + rf"\Predict-Store-Future\sales-{i}.pkl")
    sales = pd.read_pickle(OTHER + rf"\Sale-Store\sales-{i}.pkl")
    model = joblib.load(OTHER + rf"\Models\{i}_recursive_model.pkl")
    gc.collect()
    cols = ['dept_id_enc_mean','item_id_enc_mean','sales','state_id_enc_mean','cat_id_enc_mean','item_id_cat_id_enc_mean','dept_id_item_id_enc_mean','state_dept_id_enc_mean','state_item_id_enc_mean','state_item_dept_enc_mean','store_cat_id_enc_mean','store_dept_id_enc_mean','store_item_id_enc_mean','sales_lag_29','sales_lag_30','sales_lag_31','sales_lag_35','sales_lag_42','sales_lag_58','sellingTrend']

    for j in cols:
        if j in sales.columns:
            sales[j] = pd.to_numeric(sales[j])
        if j in testdata.columns:
            testdata[j] = pd.to_numeric(testdata[j])

    feature_cols = [c for c in sales.columns if c not in ("sales", "store_id")]

    preds = run_recursive_rollout(model, sales, testdata, feature_cols, DYNAMIC_COLS)
    recursive_all.append(preds.assign(store_id=i))

    del testdata, sales, model, preds
    gc.collect()

recursive_full = pd.concat(recursive_all, ignore_index=True)
recursive_full.to_pickle(OTHER + rf"\Submission\recursive_preds{SUFFIX}.pkl")
print("Recursive done:", len(recursive_full), "rows")

RECURSIVE PREDICT: CA_1
1941
RECURSIVE PREDICT: CA_2
1941
RECURSIVE PREDICT: CA_3
1941
RECURSIVE PREDICT: CA_4
1941
RECURSIVE PREDICT: TX_1
1941
RECURSIVE PREDICT: TX_2
1941
RECURSIVE PREDICT: TX_3
1941
RECURSIVE PREDICT: WI_1
1941
RECURSIVE PREDICT: WI_2
1941
RECURSIVE PREDICT: WI_3
1941
Recursive done: 853720 rows


In [4]:
import pandas as pd
recursive_val=pd.read_pickle(OTHER + r"\Results\recursive_full_preds.pkl")

In [ ]:

def to_submission_format(preds_df, suffix, horizon=HORIZON):
    df = preds_df.copy()
    df["id"] = df["item_id"].astype(str) + "_" + df["store_id"] + suffix

    day_map = {d: f"F{rank+1}" for rank, d in enumerate(sorted(df["Date"].unique()))}
    df["F"] = df["Date"].map(day_map)

    wide = df.pivot(index="id", columns="F", values="sales")
    f_cols = [f"F{i}" for i in range(1, horizon + 1)]
    wide = wide.reindex(columns=f_cols).reset_index()
    return wide

submission_recursive_eval    = to_submission_format(recursive_full, SUFFIX)
submission_recursive_val = to_submission_format(recursive_val,"_validation")


In [ ]:

sample_submission = pd.read_csv(r"C:\Users\Shree\Desktop\CODE\M5 Forecasting\Data\sample_submission.csv")
required_ids = sample_submission[sample_submission["id"].str.endswith(SUFFIX)][["id"]]
def finalize_submission(wide_df, required_ids, horizon=HORIZON):
    final = required_ids.merge(wide_df, on="id", how="left")
    f_cols = [f"F{i}" for i in range(1, horizon + 1)]
    final[f_cols] = final[f_cols].fillna(0)
    return final

final_recursive_eval    = finalize_submission(submission_recursive_eval, required_ids)
final_recursive_val=finalize_submission(submission_recursive_val,required_ids)




In [9]:
final_recursive_eval

,id,F1,F2,F3,F4,F5,F6,F7,F8,F9,...,F19,F20,F21,F22,F23,F24,F25,F26,F27,F28
0,HOBBIES_1_001_CA_1_evaluation,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
1,HOBBIES_1_002_CA_1_evaluation,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
2,HOBBIES_1_003_CA_1_evaluation,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
3,HOBBIES_1_004_CA_1_evaluation,2.0,1.0,1.0,1.0,2.0,2.0,2.0,2.0,2.0,...,2.0,2.0,2.0,2.0,1.0,2.0,2.0,2.0,2.0,2.0
4,HOBBIES_1_005_CA_1_evaluation,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30485,FOODS_3_823_WI_3_evaluation,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
30486,FOODS_3_824_WI_3_evaluation,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
30487,FOODS_3_825_WI_3_evaluation,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
30488,FOODS_3_826_WI_3_evaluation,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0


In [11]:
# ============================================================
# CELL 8: Save
# ============================================================
x=pd.read_csv(r"D:\M5 Forecasting\Submission\submission_validation.csv")
final_recursive=pd.concat([x,final_recursive_eval])
final_recursive.to_csv(OTHER + rf"\Submission\submission_recursive.csv", index=False)
print("Saved 3 submission files to Submission\\")
len(final_recursive)

Saved 3 submission files to Submission\


60980

In [12]:
final_recursive

,id,F1,F2,F3,F4,F5,F6,F7,F8,F9,...,F19,F20,F21,F22,F23,F24,F25,F26,F27,F28
0,HOBBIES_1_001_CA_1_validation,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
1,HOBBIES_1_002_CA_1_validation,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,HOBBIES_1_003_CA_1_validation,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,...,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0
3,HOBBIES_1_004_CA_1_validation,2.0,1.0,1.0,1.0,2.0,3.0,3.0,2.0,2.0,...,2.0,3.0,3.0,2.0,2.0,1.0,1.0,2.0,3.0,3.0
4,HOBBIES_1_005_CA_1_validation,1.0,1.0,1.0,1.0,1.0,1.0,2.0,1.0,1.0,...,1.0,1.0,2.0,1.0,1.0,1.0,1.0,1.0,2.0,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30485,FOODS_3_823_WI_3_evaluation,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
30486,FOODS_3_824_WI_3_evaluation,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
30487,FOODS_3_825_WI_3_evaluation,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
30488,FOODS_3_826_WI_3_evaluation,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
